# NWRD wheat-rust segmentation: CANet-B4 vs baselines

Trains every model under one protocol, picks the threshold on **val**, reports on the held-out
**test** split, benchmarks efficiency, and writes the comparison to `runs/comparison.md`.

**Kaggle setup**
1. *Add Input* → dataset **`abdur548/nwrd-patched`**.
2. *Settings* → Accelerator **GPU T4 x2** (one GPU is used; P100 may not be supported by
   Kaggle's current PyTorch build), **Internet on** (pip and ImageNet weights).
3. *Save Version* → **Save & Run All (Commit)**. It runs in the background for up to 12 h.

**If it stops at the time budget** (the log ends in `INCOMPLETE`): open the notebook, *Add Input* →
*Your Work* → this notebook's latest version, and commit again. The restore cell copies the
previous `runs/` and every run resumes where it stopped. Repeat until the log has no `INCOMPLETE`.

**Colab**: Runtime → GPU. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (🔑 sidebar);
if secrets are unavailable the notebook asks for them, and the key stays hidden. Never paste keys
into the notebook itself. Runs are written to `MyDrive/nwrd_runs` so a disconnect does not lose
them: re-running the notebook continues where it stopped. Free Colab sessions are shorter than
Kaggle's, so expect several sessions, or set `USE_DRIVE_ON_COLAB = False` for a throwaway run.

Rough cost on a T4: 1.5–2.5 h per run × 4 runs per seed (3 with `TUNE_UNET_LR = False`) × 3 seeds,
so plan on two or three sessions (Kaggle's weekly quota is 30 GPU-h).
Runs go seed by seed, so a paired comparison exists as soon as the first seed finishes.

In [ ]:
import os, sys, glob, shutil, subprocess
!pip install -q segmentation-models-pytorch==0.5.0 efficientnet_pytorch==0.7.1 albumentations kagglehub
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'enable a GPU accelerator'
os.makedirs('code/nwrd', exist_ok=True)
sys.path.insert(0, os.path.abspath('code'))

In [ ]:
# ── Experiment settings ─────────────────────────────────────────────────────
MODELS = ['canet_b4', 'unet', 'deeplabv3plus_r50']   # unet = NWRD paper baseline; add 'deeplabv3plus_b4' for the same-encoder ablation
# The shared lr (5e-5) suits fine-tuning pretrained encoders and may under-train the
# from-scratch UNet, which would flatter CANet. True adds a UNet run with its own lr; the
# report then uses whichever UNet run is better on validation. Costs ~2-3 GPU-h per seed.
TUNE_UNET_LR = True
UNET_TUNED_LR = 1e-3
if TUNE_UNET_LR and 'unet_tuned' not in MODELS:
    MODELS.append('unet_tuned')
SEEDS = [42, 43, 44]
EPOCHS = 30
NUM_WORKERS = 4   # TIME_BUDGET_HOURS is set per platform in the next cell
# Colab wipes local disk on disconnect, so runs go to Google Drive and survive to be resumed.
# Set False to keep them in /content (lost when the runtime ends).
USE_DRIVE_ON_COLAB = True

In [ ]:
# Where runs are written. On Kaggle: /kaggle/working (kept with each committed version).
# On Colab: Google Drive, so a disconnect does not lose finished runs.
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    OUT, TIME_BUDGET_HOURS = '/kaggle/working/runs', 11.0     # Kaggle kills sessions at 12 h
else:
    OUT, TIME_BUDGET_HOURS = '/content/runs', 3.0             # free Colab sessions are shorter
    if USE_DRIVE_ON_COLAB:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            OUT = '/content/drive/MyDrive/nwrd_runs'
        except Exception as e:
            print('Drive not mounted (%s) - runs stay in %s and are LOST on disconnect.' % (e, OUT))
os.makedirs(OUT, exist_ok=True)
print('runs ->', OUT, '| time budget', TIME_BUDGET_HOURS, 'h')

### Pipeline source (generated from `nwrd/` by `scripts/build_notebook.py`; edit there)

In [ ]:
%%writefile code/nwrd/__init__.py
"""Wheat rust segmentation: models, data, training, evaluation and benchmarking."""


In [ ]:
%%writefile code/nwrd/config.py
"""Experiment protocol. Every model is trained with exactly these settings."""
from dataclasses import asdict, dataclass, field


@dataclass
class Config:
    data_root: str = ''
    out_dir: str = 'runs'
    models: list = field(default_factory=lambda: ['canet_b4', 'unet', 'deeplabv3plus_r50'])
    seeds: list = field(default_factory=lambda: [42, 43, 44])
    data_seed: int = 42          # background subsample; fixed so all runs see the same patches

    img_size: int = 512
    bg_keep_ratio: float = 0.20
    epochs: int = 30
    freeze_encoder_epochs: int = 3
    batch_size: int = 8
    num_workers: int = 2
    lr: float = 5e-5
    # 'unet_tuned' only: the shared lr suits fine-tuning pretrained encoders and may
    # under-train a from-scratch UNet, so this variant gets its own.
    unet_tuned_lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    dice_weight: float = 0.7
    bce_weight: float = 0.3
    dropout: float = 0.5         # CANet head dropout; smp baselines have none
    sched_t0: int = 5
    sched_tmult: int = 2
    sched_eta_min: float = 1e-7
    amp: bool = True
    pretrained: bool = True

    # threshold chosen on val over this grid, then frozen and applied to test
    thr_lo: float = 0.05
    thr_hi: float = 0.95
    thr_step: float = 0.01

    time_budget_hours: float = 0.0   # >0: stop cleanly before a platform session limit
    limit: int = 0                   # >0: use only N patches per split (smoke tests)

    def to_dict(self):
        return asdict(self)


In [ ]:
%%writefile code/nwrd/models.py
"""Model definitions: CANet (EfficientNet-B4) and the DeepLabV3+ baselines.

Module names inside CANet match the checkpoints trained by the sprint-2 notebook, so
existing `best_model.pth` files load unchanged.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

MODELS = {
    'canet_b4': 'CANet, EfficientNet-B4 encoder (ours)',
    'deeplabv3plus_r50': 'DeepLabV3+, ResNet-50 encoder (baseline)',
    'deeplabv3plus_b4': 'DeepLabV3+, EfficientNet-B4 encoder (same-encoder ablation)',
    'unet': 'UNet, 4 encoder / 4 decoder blocks, from scratch (NWRD paper baseline)',
    'unet_tuned': 'UNet (paper architecture) with a from-scratch learning rate (config.unet_tuned_lr)',
    # Benchmark only: CANet as first implemented, running the encoder twice per forward.
    'canet_b4_twopass': 'CANet, original two-pass forward',
}


class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1, d=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class ContextFlow(nn.Module):
    """One CAM information flow: a shallow encoder-decoder at a given downsample scale."""
    def __init__(self, in_ch, out_ch, scale):
        super().__init__()
        self.scale = scale
        self.encode = ConvBNReLU(in_ch, out_ch)
        self.decode = ConvBNReLU(out_ch, out_ch)

    def forward(self, x):
        h, w = x.shape[-2:]
        xd = F.avg_pool2d(x, self.scale) if self.scale > 1 else x
        f = self.decode(self.encode(xd))
        return F.interpolate(f, (h, w), mode='bilinear', align_corners=False) if self.scale > 1 else f


class AttentionFusion(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.att = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 4, 1), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch // 4, in_ch, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.att(x)


class CAM(nn.Module):
    """Chained context aggregation: a serial dilated global flow plus parallel context
    flows at scales 2/4/8, pre-fused by concatenation and re-fused by channel attention."""
    def __init__(self, in_ch, out_ch=256):
        super().__init__()
        self.global_flow = nn.Sequential(
            ConvBNReLU(in_ch, out_ch, k=3, p=2, d=2),
            ConvBNReLU(out_ch, out_ch, k=3, p=4, d=4),
            ConvBNReLU(out_ch, out_ch, k=3, p=8, d=8))
        self.context_flows = nn.ModuleList([
            ContextFlow(in_ch, out_ch, 2),
            ContextFlow(in_ch, out_ch, 4),
            ContextFlow(in_ch, out_ch, 8)])
        self.pre_fusion = ConvBNReLU(out_ch * 4, out_ch, k=1, p=0)
        self.re_fusion = AttentionFusion(out_ch)
        self.out_conv = ConvBNReLU(out_ch, out_ch)

    def forward(self, x):
        gf = self.global_flow(x)
        cfs = [cf(x) for cf in self.context_flows]
        fused = self.pre_fusion(torch.cat([gf] + cfs, dim=1))
        return self.out_conv(self.re_fusion(fused))


class AsymmetricDecoder(nn.Module):
    """Fuses upsampled 1/32 context with reduced 1/4 low-level features."""
    def __init__(self, high_ch, low_ch, out_ch=128):
        super().__init__()
        self.low_reduce = ConvBNReLU(low_ch, 48, k=1, p=0)
        self.fuse = nn.Sequential(
            ConvBNReLU(high_ch + 48, out_ch), ConvBNReLU(out_ch, out_ch))

    def forward(self, high, low):
        high_up = F.interpolate(high, low.shape[-2:], mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([high_up, self.low_reduce(low)], dim=1))


class CANet(nn.Module):
    def __init__(self, dropout=0.3, pretrained=True, single_pass=True):
        super().__init__()
        from efficientnet_pytorch import EfficientNet
        self.encoder = (EfficientNet.from_pretrained('efficientnet-b4') if pretrained
                        else EfficientNet.from_name('efficientnet-b4'))
        self.single_pass = single_pass
        self.reduce = ConvBNReLU(1792, 512, k=1, p=0)
        self.cam = CAM(512, 256)
        self.decoder = AsymmetricDecoder(256, 32, 128)   # reduction_2: 32ch at 1/4
        self.dropout = nn.Dropout2d(dropout)
        self.head = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1))

    def forward(self, x):
        if self.single_pass:
            # reduction_6 is exactly extract_features(x); one encoder pass yields both.
            ep = self.encoder.extract_endpoints(x)
            enc, low = ep['reduction_6'], ep['reduction_2']
        else:
            enc = self.encoder.extract_features(x)
            low = self.encoder.extract_endpoints(x)['reduction_2']
        x_cam = self.cam(self.reduce(enc))
        x_dec = self.dropout(self.decoder(x_cam, low))
        return F.interpolate(self.head(x_dec), scale_factor=4, mode='bilinear', align_corners=False)


class _DoubleConv(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))


class UNet(nn.Module):
    """Plain UNet as used by the NWRD paper (Anwar et al., Sensors 2023): 4 encoder and 4
    decoder blocks, 64-1024 channels, transposed-conv upsampling, no pretraining."""
    has_pretrained_encoder = False

    def __init__(self, base=64):
        super().__init__()
        ch = [base * 2 ** i for i in range(5)]
        self.inc = _DoubleConv(3, ch[0])
        self.encoder = nn.ModuleList(_DoubleConv(ch[i], ch[i + 1]) for i in range(4))
        self.up = nn.ModuleList(nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(4)))
        self.decoder = nn.ModuleList(_DoubleConv(ch[i] * 2, ch[i]) for i in reversed(range(4)))
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for down in self.encoder:
            skips.append(down(F.max_pool2d(skips[-1], 2)))
        x = skips.pop()
        for up, dec in zip(self.up, self.decoder):
            x = dec(torch.cat([skips.pop(), up(x)], 1))
        return self.head(x)


def build_model(name, pretrained=True, dropout=0.3):
    """All models return raw logits of shape (B, 1, H, W) and expose `.encoder`."""
    if name == 'canet_b4':
        return CANet(dropout=dropout, pretrained=pretrained)
    if name == 'canet_b4_twopass':
        return CANet(dropout=dropout, pretrained=pretrained, single_pass=False)
    if name in ('unet', 'unet_tuned'):
        return UNet()
    if name in ('deeplabv3plus_r50', 'deeplabv3plus_b4'):
        import segmentation_models_pytorch as smp
        encoder = 'resnet50' if name.endswith('r50') else 'efficientnet-b4'
        return smp.DeepLabV3Plus(encoder_name=encoder,
                                 encoder_weights='imagenet' if pretrained else None,
                                 classes=1)
    raise ValueError(f'unknown model {name!r}; choose from {sorted(MODELS)}')


def load_checkpoint(model, path, map_location='cpu'):
    """Loads either a pipeline checkpoint or a sprint-2 notebook checkpoint ({'model': sd})."""
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    model.load_state_dict(ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt)
    return ckpt


In [ ]:
%%writefile code/nwrd/data.py
"""NWRD patch dataset: split discovery, background subsampling, transforms."""
import glob
import json
import os
import random

import cv2
import numpy as np
import torch

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
DATASET_DIRNAME = 'wheat_rust_patches'


def find_data_root(hint=None):
    """Returns the directory containing train/ val/ test/.

    Searches `hint`, then Kaggle's input mount, then the kagglehub cache."""
    candidates = [hint] if hint else []
    candidates += ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub')]
    for base in filter(None, candidates):
        if os.path.isdir(os.path.join(base, 'train', 'images')):
            return base
        hits = glob.glob(os.path.join(base, '**', DATASET_DIRNAME), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(
        f'no {DATASET_DIRNAME}/ found under {candidates}; attach the Kaggle dataset '
        f'abdur548/nwrd-patched or pass --data-root')


def split_dirs(root, split):
    return os.path.join(root, split, 'images'), os.path.join(root, split, 'masks')


def read_mask(path):
    return (cv2.imread(path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)


def mask_stats(root, split, cache_dir=None):
    """Per-file rust pixel counts for a split, cached as JSON (reading 8k masks is slow)."""
    cache = os.path.join(cache_dir, f'mask_stats_{split}.json') if cache_dir else None
    if cache and os.path.exists(cache):
        with open(cache) as f:
            return json.load(f)
    _, mask_dir = split_dirs(root, split)
    stats = {}
    for f in sorted(os.listdir(mask_dir)):
        m = read_mask(os.path.join(mask_dir, f))
        stats[f] = {'pos': int(m.sum()), 'total': int(m.size)}
    if cache:
        os.makedirs(cache_dir, exist_ok=True)
        with open(cache, 'w') as f:
            json.dump(stats, f)
    return stats


def select_train_files(stats, bg_keep_ratio, seed):
    """All rust-positive patches plus a seeded random fraction of background-only ones.

    The seed is the *data* seed, fixed across models so every model sees the same patches."""
    disease = sorted(f for f, s in stats.items() if s['pos'] > 0)
    background = sorted(f for f, s in stats.items() if s['pos'] == 0)
    kept = random.Random(seed).sample(background, int(len(background) * bg_keep_ratio))
    return disease + sorted(kept), {'disease': len(disease), 'bg_kept': len(kept),
                                    'bg_total': len(background)}


def pos_weight_from_stats(stats, files):
    pos = sum(stats[f]['pos'] for f in files)
    total = sum(stats[f]['total'] for f in files)
    return (total - pos) / max(pos, 1)


def train_transform(size):
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.5), A.ElasticTransform(p=0.5),
        A.Normalize(mean=MEAN, std=STD), ToTensorV2()])


def eval_transform(size):
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    return A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])


class RustDataset(torch.utils.data.Dataset):
    def __init__(self, root, split, files, transform):
        self.img_dir, self.mask_dir = split_dirs(root, split)
        self.files, self.transform = list(files), transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, f)), cv2.COLOR_BGR2RGB)
        out = self.transform(image=img, mask=read_mask(os.path.join(self.mask_dir, f)))
        return out['image'], out['mask'].unsqueeze(0), idx


In [ ]:
%%writefile code/nwrd/metrics.py
"""Pixel metrics pooled over a whole split.

The notebooks averaged IoU per batch, which scores a correctly predicted all-background
batch as 0 and depresses the curve (~0.58 vs ~0.81 pooled). Everything here pools the
confusion counts over every pixel of the split, then computes the ratios once.
"""
import numpy as np
import torch

EPS = 1e-12


def scores(tp, fp, fn, tn):
    tp, fp, fn, tn = (float(v) for v in (tp, fp, fn, tn))
    return {
        'IoU': tp / (tp + fp + fn + EPS),
        'Dice': 2 * tp / (2 * tp + fp + fn + EPS),
        'Precision': tp / (tp + fp + EPS),
        'Recall': tp / (tp + fn + EPS),
        'Specificity': tn / (tn + fp + EPS),
        'Accuracy': (tp + tn) / (tp + tn + fp + fn + EPS),
    }


def per_image_counts(probs, target, threshold):
    """(B, 4) int64 array of tp, fp, fn, tn per image. Decision rule: prob >= threshold."""
    pred = probs >= threshold
    tgt = target > 0.5
    dims = tuple(range(1, probs.dim()))
    tp = (pred & tgt).sum(dims)
    fp = (pred & ~tgt).sum(dims)
    fn = (~pred & tgt).sum(dims)
    tn = (~pred & ~tgt).sum(dims)
    return torch.stack([tp, fp, fn, tn], 1).cpu().numpy().astype(np.int64)


class ProbHistogram:
    """Histograms of predicted probability for rust and background pixels.

    Gives exact pooled confusion counts at any threshold on a 1/bins grid without keeping
    every prediction in memory."""
    def __init__(self, bins=1000):
        self.bins = bins
        self.pos = np.zeros(bins, np.int64)
        self.neg = np.zeros(bins, np.int64)

    def update(self, probs, target):
        q = (probs.float() * self.bins).long().clamp_(0, self.bins - 1).flatten()
        t = (target > 0.5).flatten()
        self.pos += torch.bincount(q[t], minlength=self.bins).cpu().numpy()
        self.neg += torch.bincount(q[~t], minlength=self.bins).cpu().numpy()

    def counts(self, threshold):
        k = int(round(threshold * self.bins))
        tp, fp = self.pos[k:].sum(), self.neg[k:].sum()
        return tp, fp, self.pos.sum() - tp, self.neg.sum() - fp

    def sweep(self, lo=0.05, hi=0.95, step=0.01):
        rows = []
        for t in np.round(np.arange(lo, hi + step / 2, step), 4):
            s = scores(*self.counts(t))
            rows.append({'threshold': float(t), **s})
        return rows

    def best_threshold(self, **kw):
        """IoU = Dice / (2 - Dice), so maximising either picks the same threshold."""
        rows = self.sweep(**kw)
        return max(rows, key=lambda r: r['IoU'])['threshold'], rows


In [ ]:
%%writefile code/nwrd/train.py
"""Training for one (model, seed) run. Resumable from last.pt."""
import json
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn

from . import data as D
from .metrics import ProbHistogram, scores
from .models import build_model


class CombinedLoss(nn.Module):
    def __init__(self, dice_w, bce_w, pos_weight):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.dice_w, self.bce_w = dice_w, bce_w
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, target):
        return self.dice_w * self.dice(logits, target) + self.bce_w * self.bce(logits, target)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _limit(files, n):
    return files[:n] if n else files


def make_loaders(cfg, cache_dir):
    """Returns loaders, data info and the train generator (reseed it every epoch)."""
    root = cfg.data_root
    train_stats = D.mask_stats(root, 'train', cache_dir)
    train_files, sel = D.select_train_files(train_stats, cfg.bg_keep_ratio, cfg.data_seed)
    train_files = _limit(train_files, cfg.limit)
    val_files = _limit(sorted(D.mask_stats(root, 'val', cache_dir)), cfg.limit)
    pos_weight = D.pos_weight_from_stats(train_stats, train_files)

    kw = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers,
              pin_memory=torch.cuda.is_available())
    g = torch.Generator()
    train_loader = torch.utils.data.DataLoader(
        D.RustDataset(root, 'train', train_files, D.train_transform(cfg.img_size)),
        shuffle=True, drop_last=True, generator=g, **kw)
    val_loader = torch.utils.data.DataLoader(
        D.RustDataset(root, 'val', val_files, D.eval_transform(cfg.img_size)), shuffle=False, **kw)
    info = {**sel, 'train_used': len(train_files), 'val': len(val_files), 'pos_weight': pos_weight}
    return train_loader, val_loader, info, g


def run_validation(model, loader, criterion, device, amp):
    model.eval()
    hist, loss_sum, n = ProbHistogram(), 0.0, 0
    with torch.no_grad():
        for imgs, masks, _ in loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=amp):
                logits = model(imgs)
            loss_sum += criterion(logits.float(), masks).item() * imgs.size(0)
            n += imgs.size(0)
            hist.update(torch.sigmoid(logits.float()), masks)
    return loss_sum / n, hist


def train_run(cfg, model_name, seed, run_dir, log=print):
    os.makedirs(run_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = cfg.amp and device.type == 'cuda'
    torch.backends.cudnn.benchmark = True
    set_seed(seed)

    train_loader, val_loader, info, gen = make_loaders(cfg, os.path.join(cfg.out_dir, '_cache'))
    log(f'[{model_name} s{seed}] data: {info}')
    model = build_model(model_name, pretrained=cfg.pretrained, dropout=cfg.dropout).to(device)
    criterion = CombinedLoss(cfg.dice_weight, cfg.bce_weight,
                             torch.tensor([info['pos_weight']], device=device))
    lr = cfg.unet_tuned_lr if model_name == 'unet_tuned' else cfg.lr
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=cfg.sched_t0, T_mult=cfg.sched_tmult, eta_min=cfg.sched_eta_min)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    history = {k: [] for k in ('train_loss', 'val_loss', 'val_iou', 'val_dice',
                               'val_iou_best_thr', 'lr', 'epoch_seconds')}
    start_epoch, best_iou = 0, -1.0
    last_path, best_path = os.path.join(run_dir, 'last.pt'), os.path.join(run_dir, 'best.pt')
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        scaler.load_state_dict(ck['scaler'])
        torch.set_rng_state(ck['rng_cpu'])
        if device.type == 'cuda' and ck.get('rng_cuda') is not None:
            torch.cuda.set_rng_state_all(ck['rng_cuda'])
        start_epoch, best_iou, history = ck['epoch'], ck['best_iou'], ck['history']
        log(f'[{model_name} s{seed}] resumed at epoch {start_epoch}')

    t_start = time.time()
    for epoch in range(start_epoch, cfg.epochs):
        if cfg.time_budget_hours and epoch > start_epoch:
            elapsed = time.time() - t_start
            per_epoch = elapsed / (epoch - start_epoch)
            if elapsed + 1.5 * per_epoch > cfg.time_budget_hours * 3600:
                log(f'[{model_name} s{seed}] time budget reached at epoch {epoch}; resume later')
                return False

        # Freezing only makes sense for an ImageNet-pretrained encoder.
        frozen = epoch < cfg.freeze_encoder_epochs and getattr(model, 'has_pretrained_encoder', True)
        for p in model.encoder.parameters():
            p.requires_grad = not frozen

        # Shuffle order and augmentation depend on (seed, epoch) only, so a resumed run
        # sees the same batches it would have seen uninterrupted.
        gen.manual_seed(seed * 10_000 + epoch)
        t0 = time.time()
        model.train()
        loss_sum, n = 0.0, 0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=amp):
                logits = model(imgs)
            loss = criterion(logits.float(), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * imgs.size(0)
            n += imgs.size(0)
        lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        val_loss, hist = run_validation(model, val_loader, criterion, device, amp)
        val = scores(*hist.counts(0.5))
        _, sweep = hist.best_threshold(lo=cfg.thr_lo, hi=cfg.thr_hi, step=cfg.thr_step)
        for k, v in (('train_loss', loss_sum / n), ('val_loss', val_loss),
                     ('val_iou', val['IoU']), ('val_dice', val['Dice']),
                     ('val_iou_best_thr', max(r['IoU'] for r in sweep)), ('lr', lr),
                     ('epoch_seconds', time.time() - t0)):
            history[k].append(v)
        log(f'[{model_name} s{seed}] ep {epoch + 1:02d}/{cfg.epochs} '
            f'{"(enc frozen) " if frozen else ""}train {loss_sum / n:.4f} val {val_loss:.4f} '
            f'IoU@0.5 {val["IoU"]:.4f} Dice@0.5 {val["Dice"]:.4f} ({history["epoch_seconds"][-1]:.0f}s)')

        # Model selection: pooled val IoU at the fixed 0.5 threshold.
        if val['IoU'] > best_iou:
            best_iou = val['IoU']
            torch.save({'model': model.state_dict(), 'epoch': epoch + 1, 'val_iou@0.5': best_iou,
                        'model_name': model_name, 'seed': seed}, best_path)
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                    'rng_cpu': torch.get_rng_state(),
                    'rng_cuda': torch.cuda.get_rng_state_all() if device.type == 'cuda' else None,
                    'epoch': epoch + 1, 'best_iou': best_iou, 'history': history}, last_path)
        with open(os.path.join(run_dir, 'history.json'), 'w') as f:
            json.dump({'history': history, 'data': info, 'best_val_iou@0.5': best_iou}, f, indent=1)

    os.remove(last_path)   # optimizer state is ~2/3 of the file; best.pt is what we keep
    with open(os.path.join(run_dir, 'TRAINED'), 'w') as f:
        f.write(f'best val IoU@0.5 {best_iou:.6f}\n')
    return True


In [ ]:
%%writefile code/nwrd/evaluate.py
"""Evaluation of a trained run.

Protocol: the binarisation threshold is chosen on the validation split, frozen, and
applied once to the held-out test split. Test metrics are also reported at the fixed 0.5
threshold, which needs no tuning at all.
"""
import json
import os

import numpy as np
import torch

from . import data as D
from .metrics import ProbHistogram, per_image_counts, scores
from .models import build_model, load_checkpoint


def predict_split(model, cfg, split, device, thresholds=()):
    """Returns the probability histogram, per-image counts at each threshold, and files."""
    files = sorted(os.listdir(D.split_dirs(cfg.data_root, split)[0]))
    files = files[:cfg.limit] if cfg.limit else files
    loader = torch.utils.data.DataLoader(
        D.RustDataset(cfg.data_root, split, files, D.eval_transform(cfg.img_size)),
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, shuffle=False)
    amp = cfg.amp and device.type == 'cuda'
    hist = ProbHistogram()
    counts = {t: [] for t in thresholds}
    model.eval()
    with torch.no_grad():
        for imgs, masks, _ in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.autocast(device.type, enabled=amp):
                probs = torch.sigmoid(model(imgs).float())
            hist.update(probs, masks)
            for t in thresholds:
                counts[t].append(per_image_counts(probs, masks, t))
    return hist, {t: np.concatenate(v) for t, v in counts.items()}, files


def evaluate_run(cfg, model_name, run_dir, log=print):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model(model_name, pretrained=False, dropout=cfg.dropout).to(device)
    ckpt = load_checkpoint(model, os.path.join(run_dir, 'best.pt'), map_location=device)

    val_hist, _, _ = predict_split(model, cfg, 'val', device)
    thr, val_sweep = val_hist.best_threshold(lo=cfg.thr_lo, hi=cfg.thr_hi, step=cfg.thr_step)
    at_edge = thr in (cfg.thr_lo, cfg.thr_hi)

    test_hist, test_counts, test_files = predict_split(model, cfg, 'test', device,
                                                       thresholds=tuple(dict.fromkeys((thr, 0.5))))
    rust = np.array([c[0] + c[2] > 0 for c in test_counts[thr]])
    res = {
        'model': model_name,
        'best_epoch': ckpt.get('epoch'),
        'threshold': thr,
        'threshold_at_grid_edge': bool(at_edge),
        'val@thr': scores(*val_hist.counts(thr)),
        'val@0.5': scores(*val_hist.counts(0.5)),
        'test@thr': scores(*test_hist.counts(thr)),
        'test@0.5': scores(*test_hist.counts(0.5)),
        # NWRD paper Table 4 protocol: background-only patches removed from the test set.
        'test_rust_patches@thr': scores(*test_counts[thr][rust].sum(0)),
        'test_images': len(test_files),
        'test_rust_images': int(rust.sum()),
        'test_rust_pixel_fraction': float(test_hist.pos.sum() / (test_hist.pos.sum() + test_hist.neg.sum())),
    }
    with open(os.path.join(run_dir, 'metrics.json'), 'w') as f:
        json.dump(res, f, indent=2)
    with open(os.path.join(run_dir, 'val_threshold_sweep.json'), 'w') as f:
        json.dump(val_sweep, f)
    # Per-image confusion counts drive the paired bootstrap in compare.py.
    np.savez_compressed(os.path.join(run_dir, 'test_counts.npz'), files=np.array(test_files),
                        at_thr=test_counts[thr], at_05=test_counts[0.5])
    t = res['test@thr']
    log(f'[{model_name} {os.path.basename(run_dir)}] thr {thr:.2f}{" (GRID EDGE)" if at_edge else ""} | '
        f'test IoU {t["IoU"]:.4f} Dice {t["Dice"]:.4f} P {t["Precision"]:.4f} R {t["Recall"]:.4f}')
    return res


def plot_qualitative(cfg, run_dirs, path, n=6, seed=0):
    """Test patches with rust: image, ground truth, then each model's prediction at its own
    val-selected threshold. `run_dirs` maps model name -> run directory."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    counts = np.load(os.path.join(next(iter(run_dirs.values())), 'test_counts.npz'))
    rust = [i for i, c in enumerate(counts['at_thr']) if c[0] + c[2] > 0]
    idx = sorted(np.random.default_rng(seed).choice(rust, min(n, len(rust)), replace=False))
    files = [str(counts['files'][i]) for i in idx]
    ds = D.RustDataset(cfg.data_root, 'test', files, D.eval_transform(cfg.img_size))
    imgs = torch.stack([ds[i][0] for i in range(len(ds))])
    masks = torch.stack([ds[i][1] for i in range(len(ds))])

    preds = {}
    for name, run_dir in run_dirs.items():
        model = build_model(name, pretrained=False, dropout=cfg.dropout).to(device).eval()
        load_checkpoint(model, os.path.join(run_dir, 'best.pt'), map_location=device)
        with open(os.path.join(run_dir, 'metrics.json')) as f:
            thr = json.load(f)['threshold']
        with torch.no_grad():
            preds[name] = (torch.sigmoid(model(imgs.to(device))).cpu() >= thr, thr)
        del model

    mean, std = torch.tensor(D.MEAN)[:, None, None], torch.tensor(D.STD)[:, None, None]
    cols = ['Image', 'Ground truth'] + [f'{k} (t={t:.2f})' for k, (_, t) in preds.items()]
    fig, axes = plt.subplots(len(files), len(cols), figsize=(3.2 * len(cols), 3.2 * len(files)),
                             squeeze=False)
    for r in range(len(files)):
        axes[r][0].imshow((imgs[r] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r][1].imshow(masks[r, 0].numpy(), cmap='gray')
        for c, (p, _) in enumerate(preds.values(), start=2):
            axes[r][c].imshow(p[r, 0].numpy(), cmap='gray')
        for c, ax in enumerate(axes[r]):
            ax.axis('off')
            if r == 0:
                ax.set_title(cols[c], fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


In [ ]:
%%writefile code/nwrd/benchmark.py
"""Inference efficiency: parameters, FLOPs, latency, throughput and peak memory.

Every model is measured in the same process on the same device with random weights
(weights do not affect cost). `canet_b4_twopass` is CANet as first implemented, running
the encoder twice per forward, so the gain from the single-pass fix is measured too.
"""
import json
import platform
import time

import torch

from .models import MODELS, build_model


def count_flops(model, size, device):
    from torch.utils.flop_counter import FlopCounterMode
    x = torch.randn(1, 3, size, size, device=device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops()


def time_forward(model, x, device, amp, warmup, iters):
    """Median wall time per forward, in milliseconds."""
    times = []
    with torch.no_grad(), torch.autocast(device.type, enabled=amp):
        for i in range(warmup + iters):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            if i >= warmup:
                times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    return times[len(times) // 2]


def benchmark_model(name, size=512, batch=8, device=None, quick=False):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model(name, pretrained=False).to(device).eval()
    params = sum(p.numel() for p in model.parameters())
    res = {'model': name, 'description': MODELS[name], 'params_M': params / 1e6,
           'GFLOPs': count_flops(model, size, device) / 1e9, 'input': f'{size}x{size}'}
    res['GMACs'] = res['GFLOPs'] / 2

    if device.type == 'cuda':
        warm, iters = (3, 5) if quick else (20, 100)
        x1 = torch.randn(1, 3, size, size, device=device)
        xb = torch.randn(batch, 3, size, size, device=device)
        res['gpu_latency_ms_bs1_fp32'] = time_forward(model, x1, device, False, warm, iters)
        res['gpu_latency_ms_bs1_fp16'] = time_forward(model, x1, device, True, warm, iters)
        torch.cuda.reset_peak_memory_stats()
        ms = time_forward(model, xb, device, True, warm, iters)
        res[f'gpu_throughput_img_s_bs{batch}_fp16'] = batch * 1000 / ms
        res[f'gpu_peak_mem_MB_bs{batch}_fp16'] = torch.cuda.max_memory_allocated() / 2**20

    cpu = torch.device('cpu')
    model = model.to(cpu)
    warm, iters = (1, 2) if quick else (3, 10)
    res['cpu_latency_ms_bs1_fp32'] = time_forward(
        model, torch.randn(1, 3, size, size), cpu, False, warm, iters)
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return res


def run_benchmarks(models, out_path, size=512, quick=False, log=print):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    env = {'device': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
           'cpu': platform.processor() or platform.machine(),
           'cpu_threads': torch.get_num_threads(), 'torch': torch.__version__}
    rows = []
    for name in models:
        r = benchmark_model(name, size=size, device=device, quick=quick)
        rows.append(r)
        log(f'[bench] {name}: {r["params_M"]:.2f}M params, {r["GFLOPs"]:.1f} GFLOPs, '
            + ', '.join(f'{k} {v:.1f}' for k, v in r.items() if k.startswith(('gpu_', 'cpu_'))))
    with open(out_path, 'w') as f:
        json.dump({'env': env, 'results': rows}, f, indent=2)
    return rows


In [ ]:
%%writefile code/nwrd/compare.py
"""Aggregates finished runs into the final comparison.

Outputs in <out_dir>/: comparison.md, comparison.json, figures/*.png, and export/ with the
files the FastAPI backend and dashboard serve.
"""
import glob
import json
import os
import shutil

import numpy as np

from .models import MODELS

METRICS = ('IoU', 'Dice', 'Precision', 'Recall', 'Specificity', 'Accuracy')

# Published NWRD paper results (Anwar et al., Sensors 2023, doi:10.3390/s23156942), UNet + APF.
# Evaluated on 128px patches of 10 held-out full images; their IoU and F1 are not mutually
# consistent (likely per-image averaging), so treat them as approximate reference points.
PAPER = [
    ('Table 3, full test set', {'Precision': 0.506, 'Recall': 0.624, 'Dice': 0.557}),
    ('Table 4, background-only patches removed', {'Precision': 0.593, 'Recall': 0.552,
                                                  'Dice': 0.564, 'IoU': 0.438}),
]
OURS = 'canet_b4'


def load_runs(out_dir):
    runs = {}
    for path in sorted(glob.glob(os.path.join(out_dir, '*', 'seed*', 'metrics.json'))):
        run_dir = os.path.dirname(path)
        model, seed = os.path.basename(os.path.dirname(run_dir)), int(os.path.basename(run_dir)[4:])
        with open(path) as f:
            m = json.load(f)
        with open(os.path.join(run_dir, 'history.json')) as f:
            h = json.load(f)
        c = np.load(os.path.join(run_dir, 'test_counts.npz'))
        runs.setdefault(model, {})[seed] = {'dir': run_dir, 'metrics': m, 'history': h,
                                            'files': c['files'], 'counts': c['at_thr']}
    return runs


def _iou(c):
    tp, fp, fn = c[..., 0], c[..., 1], c[..., 2]
    return tp / np.maximum(tp + fp + fn, 1)


def _dice(c):
    tp, fp, fn = c[..., 0], c[..., 1], c[..., 2]
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def paired_bootstrap(ours, base, seeds, n_boot=5000, rng_seed=0):
    """Image-level paired bootstrap of the seed-averaged difference in pooled test IoU/Dice.

    Each resample draws test images with replacement; the same images are used for both
    models and every seed, so the interval reflects test-set sampling uncertainty."""
    for s in seeds:
        assert (ours[s]['files'] == base[s]['files']).all(), 'test file order differs'
    n = len(ours[seeds[0]]['files'])
    w = np.random.default_rng(rng_seed).multinomial(n, np.full(n, 1 / n), size=n_boot)
    out = {}
    for name, fn in (('IoU', _iou), ('Dice', _dice)):
        d = np.mean([fn(w @ ours[s]['counts']) - fn(w @ base[s]['counts']) for s in seeds], 0)
        point = np.mean([fn(ours[s]['counts'].sum(0)) - fn(base[s]['counts'].sum(0)) for s in seeds])
        out[name] = {'delta': float(point), 'ci95': [float(np.percentile(d, 2.5)),
                                                     float(np.percentile(d, 97.5))],
                     'p_two_sided': float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))}
    return out


def seed_ttest(ours, base, seeds):
    if len(seeds) < 2:
        return None
    from scipy import stats
    a = [ours[s]['metrics']['test@thr']['IoU'] for s in seeds]
    b = [base[s]['metrics']['test@thr']['IoU'] for s in seeds]
    t, p = stats.ttest_rel(a, b)
    return {'t': float(t), 'p': float(p), 'n_seeds': len(seeds)}


def _mean_std(vals):
    vals = np.asarray(vals, float)
    return float(vals.mean()), float(vals.std(ddof=1)) if len(vals) > 1 else 0.0


def _fmt(m, s, n):
    return f'{m:.4f} ± {s:.4f}' if n > 1 else f'{m:.4f}'


def summarise(runs):
    summary = {}
    for model, seeds in runs.items():
        ms = [seeds[s]['metrics'] for s in sorted(seeds)]
        hs = [seeds[s]['history']['history'] for s in sorted(seeds)]
        row = {'seeds': sorted(seeds), 'n': len(ms)}
        for split in ('test@thr', 'test@0.5', 'val@thr', 'test_rust_patches@thr'):
            row[split] = {k: _mean_std([m[split][k] for m in ms]) for k in METRICS}
        row['threshold'] = _mean_std([m['threshold'] for m in ms])
        row['threshold_at_grid_edge'] = any(m['threshold_at_grid_edge'] for m in ms)
        row['best_epoch'] = [m['best_epoch'] for m in ms]
        row['train_min_per_epoch'] = _mean_std([np.mean(h['epoch_seconds']) / 60 for h in hs])
        summary[model] = row
    return summary


def write_markdown(path, summary, tests, bench, test_info):
    order = [m for m in (OURS,) if m in summary] + sorted(m for m in summary if m != OURS)
    L = ['# CANet-B4 vs baselines: final comparison', '',
         f'Test split: {test_info["test_images"]} patches, {test_info["test_rust_images"]} with rust, '
         f'{100 * test_info["test_rust_pixel_fraction"]:.1f}% rust pixels. Threshold chosen per run on '
         'val (IoU-maximising, grid 0.05–0.95), then frozen for test. Metrics pooled over all test '
         'pixels; mean ± sample std over seeds.', '',
         '## Accuracy (test split, val-selected threshold)', '',
         '| Model | Seeds | ' + ' | '.join(METRICS) + ' | IoU @0.5 | Threshold |',
         '|---|---|' + '---|' * (len(METRICS) + 2)]
    for m in order:
        r = summary[m]
        cells = [_fmt(*r['test@thr'][k], r['n']) for k in METRICS]
        thr = f'{r["threshold"][0]:.2f}' + (' ⚠ edge' if r['threshold_at_grid_edge'] else '')
        L.append(f'| {"**" + m + "** (ours)" if m == OURS else m} | {r["n"]} | ' + ' | '.join(cells)
                 + f' | {_fmt(*r["test@0.5"]["IoU"], r["n"])} | {thr} |')
    L += ['', '## Against the NWRD paper (UNet + APF)', '',
          '| Result | Precision | Recall | F1 / Dice | IoU |', '|---|---|---|---|---|']
    for name, p in PAPER:
        L.append(f'| Paper, {name} | ' + ' | '.join(f'{p[k]:.3f}' if k in p else '–'
                                                     for k in ('Precision', 'Recall', 'Dice', 'IoU')) + ' |')
    for m in order:
        r = summary[m]
        for split, label in (('test@thr', 'all test patches'),
                             ('test_rust_patches@thr', 'rust-positive test patches (≈ Table 4)')):
            L.append(f'| {m}, re-run: {label} | ' + ' | '.join(
                f'{r[split][k][0]:.3f}' for k in ('Precision', 'Recall', 'Dice', 'IoU')) + ' |')
    L += ['', 'Paper figures are as published, on a different test unit (full-image patches, '
          "128px). The `unet` row is the paper's architecture re-trained under this protocol, so "
          'CANet vs `unet` is the like-for-like comparison.']
    if tests:
        L += ['', '## Is the difference real? (CANet-B4 minus baseline, test split)', '',
              '| Baseline | Paired seeds | ΔIoU [95% CI] | p (bootstrap) | ΔDice [95% CI] | p | seed-level t-test p |',
              '|---|---|---|---|---|---|---|']
        for base, t in tests.items():
            bi, bd, tt = t['bootstrap']['IoU'], t['bootstrap']['Dice'], t['seed_ttest']
            L.append(f'| {base} | {len(t["seeds"])} | {bi["delta"]:+.4f} [{bi["ci95"][0]:+.4f}, '
                     f'{bi["ci95"][1]:+.4f}] | {bi["p_two_sided"]:.4f} | {bd["delta"]:+.4f} '
                     f'[{bd["ci95"][0]:+.4f}, {bd["ci95"][1]:+.4f}] | {bd["p_two_sided"]:.4f} | '
                     f'{"%.4f" % tt["p"] if tt else "n/a (1 seed)"} |')
        unets = [m for m in ('unet', 'unet_tuned') if m in summary]
        if len(unets) == 2:
            best = max(unets, key=lambda m: summary[m]['val@thr']['IoU'][0])
            L += ['', f'Two runs of the paper\'s UNet exist (shared lr and `unet_tuned`). The '
                  f'stronger on **validation** IoU, `{best}`, is the paper-architecture baseline; '
                  'choosing by test IoU would bias the comparison.']
        L += ['', 'A CI that excludes 0 means the gap is larger than test-set sampling noise. '
              'The bootstrap does not capture training-run variance; the seed-level t-test does, '
              'but has little power with 3 seeds.']
    if bench:
        env, rows = bench['env'], {r['model']: r for r in bench['results']}
        gpu_keys = [k for k in next(iter(rows.values())) if k.startswith('gpu_')]
        L += ['', f'## Efficiency ({env["device"]}, {next(iter(rows.values()))["input"]} input, '
              f'torch {env["torch"]})', '',
              '| Model | Params (M) | GFLOPs | ' + ' | '.join(k[4:] for k in gpu_keys)
              + ' | CPU latency bs1 (ms) | Train min/epoch |',
              '|---|---|---|' + '---|' * (len(gpu_keys) + 2)]
        for m in [x for x in dict.fromkeys(order + list(rows)) if x in rows]:
            r = rows[m]
            tr = summary.get(m, {}).get('train_min_per_epoch')
            L.append(f'| {m} | {r["params_M"]:.2f} | {r["GFLOPs"]:.1f} | '
                     + ' | '.join(f'{r[k]:.1f}' for k in gpu_keys)
                     + f' | {r["cpu_latency_ms_bs1_fp32"]:.0f} | {f"{tr[0]:.1f}" if tr else "–"} |')
        if 'canet_b4_twopass' in rows:
            L += ['', '`canet_b4_twopass` is CANet as originally implemented (encoder run twice per '
                  'forward). Outputs are bit-identical to `canet_b4`; only the cost differs.']
    L += ['', '## Models', ''] + [f'- `{m}` — {MODELS[m]}' for m in order]
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(L) + '\n')


def plot_figures(runs, fig_dir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    os.makedirs(fig_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for model, seeds in sorted(runs.items()):
        for key, ax in (('val_loss', axes[0]), ('val_iou', axes[1])):
            curves = [seeds[s]['history']['history'][key] for s in sorted(seeds)]
            n = min(map(len, curves))
            arr = np.array([c[:n] for c in curves])
            x = np.arange(1, n + 1)
            line, = ax.plot(x, arr.mean(0), label=model, lw=2 if model == OURS else 1.4)
            if len(curves) > 1:
                ax.fill_between(x, arr.min(0), arr.max(0), color=line.get_color(), alpha=0.15)
    axes[0].set_title('Validation loss'); axes[1].set_title('Validation IoU (pooled, threshold 0.5)')
    for ax in axes:
        ax.set_xlabel('Epoch'); ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(fig_dir, 'training_curves.png'), dpi=150)
    plt.close(fig)


def plot_confusion(counts, thr, path):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    tp, fp, fn, tn = counts.sum(0)
    cm = np.array([[tn, fp], [fn, tp]])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f'{v:,}', ha='center', va='center',
                color='white' if v > cm.max() / 2 else 'black')
    ax.set_xticks([0, 1], ['Pred background', 'Pred rust'])
    ax.set_yticks([0, 1], ['True background', 'True rust'])
    ax.set_title(f'Test confusion matrix (threshold {thr:.2f})')
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def export_for_app(runs, out_dir, fig_dir):
    """Picks the CANet seed with the best *validation* IoU and writes the files the backend
    serves. Selection never looks at test metrics."""
    if OURS not in runs:
        return None
    seed = max(runs[OURS], key=lambda s: runs[OURS][s]['metrics']['val@thr']['IoU'])
    run = runs[OURS][seed]
    m, h = run['metrics'], run['history']['history']
    exp = os.path.join(out_dir, 'export')
    os.makedirs(exp, exist_ok=True)
    t = m['test@thr']
    results = {
        'model': OURS, 'seed': seed, 'split': 'test', 'best_epoch': m['best_epoch'],
        'best_threshold': m['threshold'],
        'final_metrics': {'IoU': round(t['IoU'], 4), 'F1 / Dice': round(t['Dice'], 4),
                          'Precision': round(t['Precision'], 4), 'Recall': round(t['Recall'], 4),
                          'Specificity': round(t['Specificity'], 4),
                          'Accuracy': round(t['Accuracy'], 4)},
        'history': {k: h[k] for k in ('train_loss', 'val_loss', 'val_iou', 'val_dice')},
    }
    with open(os.path.join(exp, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2)
    shutil.copy(os.path.join(run['dir'], 'best.pt'), os.path.join(exp, 'best_model.pth'))
    plot_confusion(run['counts'], m['threshold'], os.path.join(exp, 'confusion_matrix.png'))
    for name in ('training_curves.png', 'qualitative_results.png'):
        if os.path.exists(os.path.join(fig_dir, name)):
            shutil.copy(os.path.join(fig_dir, name), os.path.join(exp, name))
    return seed


def compare(out_dir, log=print):
    runs = load_runs(out_dir)
    if not runs:
        log('[compare] no finished runs yet'); return None
    summary = summarise(runs)
    tests = {}
    if OURS in runs:
        for base in (m for m in runs if m != OURS):
            seeds = sorted(set(runs[OURS]) & set(runs[base]))
            if seeds:
                tests[base] = {'seeds': seeds,
                               'bootstrap': paired_bootstrap(runs[OURS], runs[base], seeds),
                               'seed_ttest': seed_ttest(runs[OURS], runs[base], seeds)}
    bench_path = os.path.join(out_dir, 'benchmark.json')
    bench = json.load(open(bench_path)) if os.path.exists(bench_path) else None
    any_run = next(iter(next(iter(runs.values())).values()))['metrics']
    fig_dir = os.path.join(out_dir, 'figures')
    plot_figures(runs, fig_dir)
    write_markdown(os.path.join(out_dir, 'comparison.md'), summary, tests, bench, any_run)
    exported = export_for_app(runs, out_dir, fig_dir)
    with open(os.path.join(out_dir, 'comparison.json'), 'w') as f:
        json.dump({'summary': summary, 'significance': tests, 'benchmark': bench,
                   'exported_seed': exported}, f, indent=2)
    log(open(os.path.join(out_dir, 'comparison.md'), encoding='utf-8').read())
    return summary


In [ ]:
%%writefile code/nwrd/run.py
"""Runs the full experiment: train -> evaluate -> benchmark -> figures -> compare.

    python -m nwrd.run --data-root /path/to/wheat_rust_patches --out runs
    python -m nwrd.run --stage compare --out runs        # re-aggregate only

Finished stages are skipped, and an interrupted training run resumes from last.pt, so the
same command can be repeated across Kaggle sessions until everything is done.
"""
import argparse
import json
import os
import sys
import time
from dataclasses import fields

from .config import Config

STAGES = ('train', 'eval', 'bench', 'figures', 'compare')


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--stage', default='all', choices=('all',) + STAGES)
    p.add_argument('--out', dest='out_dir', default=Config.out_dir)
    p.add_argument('--bench-models', nargs='+', default=None,
                   help='default: the trained models plus canet_b4_twopass')
    p.add_argument('--bench-quick', action='store_true')
    for f in fields(Config):
        if f.name == 'out_dir':
            continue
        flag = '--' + f.name.replace('_', '-')
        default = f.default_factory() if callable(f.default_factory) else f.default
        if isinstance(default, bool):
            p.add_argument(flag, dest=f.name, type=lambda s: s.lower() in ('1', 'true', 'yes'),
                           default=default)
        elif isinstance(default, list):
            p.add_argument(flag, dest=f.name, nargs='+', type=type(default[0]), default=default)
        else:
            p.add_argument(flag, dest=f.name, type=type(default), default=default)
    return p.parse_args(argv)


def main(argv=None):
    a = parse_args(argv)
    cfg = Config(**{f.name: getattr(a, f.name) for f in fields(Config)})
    os.makedirs(cfg.out_dir, exist_ok=True)
    log_file = open(os.path.join(cfg.out_dir, 'log.txt'), 'a', encoding='utf-8')

    def log(msg):
        line = f'{time.strftime("%H:%M:%S")} {msg}'
        print(line, flush=True)
        log_file.write(line + '\n'); log_file.flush()

    stages = STAGES if a.stage == 'all' else (a.stage,)
    needs_data = {'train', 'eval', 'figures'} & set(stages)
    if needs_data:
        from .data import find_data_root
        cfg.data_root = find_data_root(cfg.data_root or None)
        log(f'data: {cfg.data_root}')
    with open(os.path.join(cfg.out_dir, 'config.json'), 'w') as f:
        json.dump(cfg.to_dict(), f, indent=2)

    # Seed-major order: after each seed finishes, every model has a paired run to compare.
    runs = [(m, s, os.path.join(cfg.out_dir, m, f'seed{s}')) for s in cfg.seeds for m in cfg.models]
    complete = True
    for model, seed, run_dir in runs:
        if 'train' in stages and not os.path.exists(os.path.join(run_dir, 'TRAINED')):
            from .train import train_run
            if not train_run(cfg, model, seed, run_dir, log=log):
                complete = False
                break          # out of time budget: stop here, resume next session
        if 'eval' in stages and os.path.exists(os.path.join(run_dir, 'TRAINED')) \
                and not os.path.exists(os.path.join(run_dir, 'metrics.json')):
            from .evaluate import evaluate_run
            evaluate_run(cfg, model, run_dir, log=log)

    if 'bench' in stages and not os.path.exists(os.path.join(cfg.out_dir, 'benchmark.json')):
        from .benchmark import run_benchmarks
        models = a.bench_models or list(dict.fromkeys(cfg.models + ['canet_b4_twopass']))
        run_benchmarks(models, os.path.join(cfg.out_dir, 'benchmark.json'), size=cfg.img_size,
                       quick=a.bench_quick, log=log)

    if 'figures' in stages:
        from .evaluate import plot_qualitative
        first = {m: os.path.join(cfg.out_dir, m, f'seed{cfg.seeds[0]}') for m in cfg.models}
        first = {m: d for m, d in first.items() if os.path.exists(os.path.join(d, 'metrics.json'))}
        if first:
            os.makedirs(os.path.join(cfg.out_dir, 'figures'), exist_ok=True)
            plot_qualitative(cfg, first, os.path.join(cfg.out_dir, 'figures', 'qualitative_results.png'))

    if 'compare' in stages:
        from .compare import compare
        compare(cfg.out_dir, log=log)
    if not complete:
        log('INCOMPLETE: re-run the same command in a new session to continue.')
    return 0 if complete else 3


if __name__ == '__main__':
    sys.exit(main())


### Data and resume

In [ ]:
from nwrd.data import find_data_root

# Kaggle: the attached dataset already sits under /kaggle/input. Colab: download it.
# Deciding by "does /kaggle/input exist" is unreliable - some Colab runtimes have an empty
# /kaggle/input - so try to find the data first and only download if that fails.
try:
    DATA_ROOT = find_data_root(None)
except FileNotFoundError:
    import kagglehub
    if not os.environ.get('KAGGLE_KEY'):
        try:
            from google.colab import userdata      # Colab: key icon in the left sidebar
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        except Exception as e:
            import getpass
            print('Colab secrets unavailable (%s); enter Kaggle credentials.' % e)
            os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ').strip()
            os.environ['KAGGLE_KEY'] = getpass.getpass('Kaggle API key (hidden): ').strip()
    DATA_ROOT = find_data_root(kagglehub.dataset_download('abdur548/nwrd-patched'))

print('dataset:', DATA_ROOT, '|', {s: len(os.listdir(os.path.join(DATA_ROOT, s, 'images'))) for s in ('train', 'val', 'test')})
# How were the splits made? If patches of one field image appear in more than one split, the
# test scores are inflated by leakage. Inspect the dataset's own notes and file naming.
for f in ('README.md', 'processing_stats.json'):
    p = os.path.join(DATA_ROOT, f)
    if os.path.exists(p):
        print('--- %s ---' % f); print(open(p, encoding='utf-8', errors='replace').read()[:3000])
for s in ('train', 'val', 'test'):
    print(s, sorted(os.listdir(os.path.join(DATA_ROOT, s, 'images')))[:5])

In [ ]:
# Continue an earlier session. On Drive/Kaggle-working, finished runs are already in OUT.
prev = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/runs/config.json', recursive=True)]
if prev and not os.path.exists(os.path.join(OUT, 'config.json')):
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True)
    print('restored previous runs from', prev[0])
done = sorted(glob.glob(os.path.join(OUT, '*', 'seed*')))
for d in done:
    state = ('done' if os.path.exists(os.path.join(d, 'metrics.json')) else
             'trained' if os.path.exists(os.path.join(d, 'TRAINED')) else
             'partial' if os.path.exists(os.path.join(d, 'last.pt')) else 'empty')
    print(' ', os.path.relpath(d, OUT), state)
print('%d run directories found' % len(done))

### Train → evaluate → benchmark → compare

In [ ]:
cmd = [sys.executable, '-m', 'nwrd.run', '--data-root', DATA_ROOT, '--out', OUT,
       '--models', *MODELS, '--seeds', *map(str, SEEDS), '--epochs', str(EPOCHS),
       '--time-budget-hours', str(TIME_BUDGET_HOURS), '--num-workers', str(NUM_WORKERS),
       '--unet-tuned-lr', str(UNET_TUNED_LR)]
print(' '.join(cmd))
proc = subprocess.Popen(cmd, cwd='code', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
print('exit code', proc.wait(), '(3 = stopped at time budget; commit again to resume)')

In [ ]:
from IPython.display import Markdown, Image, display
if os.path.exists(f'{OUT}/comparison.md'):
    display(Markdown(open(f'{OUT}/comparison.md', encoding='utf-8').read()))
    for f in ('training_curves.png', 'qualitative_results.png'):
        if os.path.exists(f'{OUT}/figures/{f}'):
            display(Image(f'{OUT}/figures/{f}'))

## Taking the results back to the repo

Download the notebook output and copy:

| From `runs/` | To the repo |
|---|---|
| `comparison.md`, `comparison.json`, `benchmark.json` | `results/` |
| `export/results.json` | `backend/results/results.json` |
| `export/best_model.pth` | `backend/models/best_model.pth` |
| `export/*.png` | `backend/visualisations/` |

`export/` holds the CANet seed with the best **validation** IoU; test metrics are never used
to choose it.